In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [19]:
order_products = pd.read_csv("../outputs/master_customer_table.csv")
personas = pd.read_csv("../outputs/customer_personas.csv")

In [20]:
order_products = order_products.merge(
    personas[["customer_unique_id", "persona"]],
    on="customer_unique_id",
    how="left"
)

order_products = order_products.dropna(
    subset=["product_category_name", "persona"]
)

order_products.head()

,customer_unique_id,order_id,order_purchase_timestamp,order_status,product_id,price,review_score,review_comment_message,product_category_name,persona
4,345ecd01c38d18a9036ed96c73b8d066,6b7d50bd145f6fc7f33cebabd7e49d0f,2018-07-29 09:51:30,delivered,9391a573abe00141c56e38d84d7d5b3b,230.0,5.0,O baratheon è esxelente Amo adoro o baratheon,casa_conforto,Silent Loyal Customers
5,4c93744516667ad3b8f1fb645a3116a4,5741ea1f91b5fbab2bd2dc653a5b5099,2017-09-14 18:14:31,delivered,0be701e03657109a8a4d5168122777fb,259.9,5.0,Loja responsável,esporte_lazer,Silent Loyal Customers
8,1175e95fb47ddff9de6b2b06188f7e0d,1ebeea841c590e86a14a0d7a48e7d062,2018-01-18 12:35:44,delivered,08401f48b67a852d06105725c3013eff,25.0,5.0,chegou antes do prometido,brinquedos,Emerging Loyalists
9,9afe194fb833f79e300e37e580171f22,7433cbcc783205509d66a5260da5b574,2018-01-08 11:22:34,delivered,78efe838c04bbc568be034082200ac20,99.9,5.0,"ótimo, entregou antes da data prevista.",moveis_decoracao,High-Value Enthusiasts
10,2a7745e1ed516b289ed9b29c7d0539a5,8428e578bb1cf839ae26a6b7615502b9,2017-11-27 17:23:20,delivered,6b874ff4c71d18e91079b40396bf176a,25.3,5.0,Td certo.Produto e prazo de entrega.,automotivo,Silent Loyal Customers


In [21]:
persona_baskets = (
    order_products
    .groupby(["persona", "order_id"])["product_category_name"]
    .apply(list)
    .reset_index()
)

persona_baskets.head()

,persona,order_id,product_category_name
0,Dissatisfied / Churn-Risk Customers,003edccf16bc5ec447f592913b3df2b4,[telefonia]
1,Dissatisfied / Churn-Risk Customers,0070092bb6004faafa287e5fc35971e9,[brinquedos]
2,Dissatisfied / Churn-Risk Customers,00a4d94df1c9757661679b2f4bd6d311,[automotivo]
3,Dissatisfied / Churn-Risk Customers,00d0ff2017f9ea13dd7eec7470ca473a,[beleza_saude]
4,Dissatisfied / Churn-Risk Customers,0136390286be8a34efdc801ed150568f,"[automotivo, automotivo]"


In [22]:
persona_baskets["basket_size"] = persona_baskets["product_category_name"].apply(len)

persona_baskets["basket_size"].value_counts()

basket_size
1     35264
2      3872
3       731
4       310
6       111
5       102
7        14
8         6
10        4
12        3
11        3
9         1
21        1
13        1
14        1
Name: count, dtype: int64

In [23]:
persona_baskets = persona_baskets[
    persona_baskets["basket_size"] >= 2
].drop(columns="basket_size")

persona_baskets.shape

(5160, 3)

In [24]:
def run_apriori(baskets):
    te = TransactionEncoder()
    te_array = te.fit(baskets).transform(baskets)
    df = pd.DataFrame(te_array, columns=te.columns_)
    
    frequent_itemsets = apriori(
        df,
        min_support=0.002,
        use_colnames=True,
        max_len=2
    )
    
    if frequent_itemsets.empty:
        return pd.DataFrame()
    
    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=0.25
    )
    
    return rules

In [25]:
persona_rules = []

top_personas = persona_baskets["persona"].value_counts().head(4).index

for persona in top_personas:
    baskets = persona_baskets[
        persona_baskets["persona"] == persona
    ]["product_category_name"].tolist()

    print(f"Persona: {persona} | Multi-item Orders: {len(baskets)}")

    rules = run_apriori(baskets)

    if not rules.empty:
        print(f"  → Rules found: {len(rules)}")
        rules["persona"] = persona
        persona_rules.append(rules)
    else:
        print("  → No rules generated")

Persona: Silent Loyal Customers | Multi-item Orders: 3158
  → Rules found: 1
Persona: High-Value Enthusiasts | Multi-item Orders: 854
  → Rules found: 3
Persona: Emerging Loyalists | Multi-item Orders: 832
  → Rules found: 3
Persona: Fading Customers | Multi-item Orders: 214
  → Rules found: 6


In [26]:
final_rules = pd.concat(persona_rules, ignore_index=True)

final_rules = final_rules[
    ["persona", "antecedents", "consequents", "support", "confidence", "lift"]
].sort_values(
    by=["persona", "lift", "confidence"],
    ascending=False
)

final_rules.head(10)

,persona,antecedents,consequents,support,confidence,lift
0,Silent Loyal Customers,(casa_conforto),(cama_mesa_banho),0.005700,0.562500,3.144027
1,High-Value Enthusiasts,(artes),(moveis_decoracao),0.002342,0.666667,4.950725
2,High-Value Enthusiasts,(casa_conforto),(cama_mesa_banho),0.004684,0.500000,3.713043
3,High-Value Enthusiasts,(construcao_ferramentas_iluminacao),(moveis_decoracao),0.002342,0.500000,3.713043
12,Fading Customers,(moveis_sala),(moveis_escritorio),0.004673,1.000000,30.571429
8,Fading Customers,(bebes),(brinquedos),0.004673,0.333333,10.190476
7,Fading Customers,(artes),(moveis_decoracao),0.004673,1.000000,6.687500
10,Fading Customers,(casa_construcao),(moveis_decoracao),0.004673,1.000000,6.687500
9,Fading Customers,(casa_conforto),(cama_mesa_banho),0.004673,0.500000,3.343750
11,Fading Customers,(construcao_ferramentas_iluminacao),(moveis_decoracao),0.004673,0.500000,3.343750


In [27]:
final_rules.to_csv(
    "../outputs/persona_apriori_rules.csv",
    index=False
)

print("persona_apriori_rules.csv saved successfully.")

persona_apriori_rules.csv saved successfully.
